In [10]:
import pandas as pd
import yaml
from pathlib import Path


def load_yaml(yaml_path):
    with open(yaml_path, "r") as f:
        return yaml.safe_load(f)


def build_mapping(yaml_data, collection):
    mapping = {}

    fields = yaml_data.get("fields", {})

    for arch_field, field_info in fields.items():
        collections = field_info.get("Collections", {})

        if collection in collections:
            fedora_fields = collections[collection]

            for fedora_field in fedora_fields:
                mapping[fedora_field] = arch_field

    return mapping


def transform_csv(csv_path, yaml_data, output_dir):

    collection = csv_path.stem
    mapping = build_mapping(yaml_data, collection)

    if not mapping:
        return "no_mapping", collection

    df = pd.read_csv(csv_path)

    cols = [c for c in df.columns if c in mapping]

    if not cols:
        return "no_columns", collection

    df_out = df[cols].rename(columns=mapping)

    output_file = output_dir / f"{collection}_ami.csv"
    df_out.to_csv(output_file, index=False)

    return "processed", collection


def generate_all(csv_folder, yaml_file, output_folder):

    csv_folder = Path(csv_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(exist_ok=True)

    yaml_data = load_yaml(yaml_file)

    processed = 0

    skipped_no_mapping = []
    skipped_no_columns = []

    for csv_file in csv_folder.glob("*.csv"):

        result, name = transform_csv(csv_file, yaml_data, output_folder)

        if result == "processed":
            processed += 1

        elif result == "no_mapping":
            skipped_no_mapping.append(name)

        elif result == "no_columns":
            skipped_no_columns.append(name)

    print("\nSummary")
    print("------------------")
    print(f"Processed files: {processed}")

    print(f"\nSkipped (no YAML mapping): {len(skipped_no_mapping)}")
    for f in skipped_no_mapping:
        print(f"  - {f}")

    print(f"\nSkipped (no matching columns): {len(skipped_no_columns)}")
    for f in skipped_no_columns:
        print(f"  - {f}")

    total_skipped = len(skipped_no_mapping) + len(skipped_no_columns)
    print(f"\nTotal skipped: {total_skipped}")


if __name__ == "__main__":

    YAML_FILE = "fedora.yml"
    CSV_FOLDER = "csv_data_files"
    OUTPUT_FOLDER = "ami_output"

    generate_all(CSV_FOLDER, YAML_FILE, OUTPUT_FOLDER)


Summary
------------------
Processed files: 32

Skipped (no YAML mapping): 141
  - berger_cloonan_first_58-2020-07-30
  - scifi-exhibit-2021-pass-d%5B2%5D
  - scifi-exhibit-2021-pass-c_5B7_5D
  - londonmaps_public_domain-2020-04-17_5B3_5D
  - londonmaps_public_domain-2020-04-15%5B4%5D
  - berger_cloonan_batch_5
  - berger_cloonan_first_58-2020-11-11_5B2_5D
  - scifi-exhibit-2021%5B7%5D
  - charting-texas-revised
  - scifi-exhibit-2021-pass-c%5B6%5D
  - scifi-exhibit-2021-pass-d_5B3_5D
  - berger_cloonan_batch_7
  - henry-test
  - londonmaps_public_domain-2020-04-17%5B2%5D
  - berger_cloonan_first_58-2020-11-11%5B3%5D
  - berger_cloonan_batch_6
  - berger_cloonan_public_domain-2020-07-30
  - scifi-exhibit-2021_5B6_5D
  - scifi-exhibit-2021-pass-c%5B8%5D
  - cherokee-freedmen-fixed
  - james-test-ingest-20251021
  - scifi-exhibit-2021-pass-c%5B4%5D
  - scifi-exhibit-2021-b%5B10%5D
  - basbanes-texts_5B2_5D
  - mark-kevin-james-test-collection-20240924
  - scifi-exhibit-2021_5B4_5D
  - s